# Titanic Multivariate Analysis

this notebook uses the datasets generated by `data/gen_multi.ipynb`. run that notebook first if the CSV files in `data/` are missing.

In [ ]:
import numpy as np
import pandas as pd

from missingfcup import MissingData
from missingfcup import Panel

from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

import plotly.io as pio
pio.renderers.default = "notebook"

In [ ]:
df_mcar = pd.read_csv("data/mcar_multi.csv")
df_mar  = pd.read_csv("data/mar_multi.csv")
df_mnar = pd.read_csv("data/mnar_multi.csv")

In [ ]:
md_mcar = MissingData(df_mcar)
md_mar = MissingData(df_mar)
md_mnar = MissingData(df_mnar)

## Exploratory Discovery

these first plots help validate the missing values distribution across the datasets.

because of how each generation method works, not all datasets share the same missing rate or the same number of missing columns. this is expected, and we will focus on proving each mechanism using visualizations.

In [ ]:
panel = Panel(
    plots=[
        md_mcar.barchart_overall_missingness(title="MCAR"),
        md_mar.barchart_overall_missingness(title="MAR"),
        md_mnar.barchart_overall_missingness(title="MNAR")
        ]
    )

panel.show()

In [ ]:
panel = Panel(
    plots=[
        md_mcar.barchart_missing_count(title="MCAR"),
        md_mar.barchart_missing_count(title="MAR"),
        md_mnar.barchart_missing_count(title="MNAR")
        ]
    )

panel.show()

In [ ]:
panel = Panel(
    plots=[
        md_mcar.heatmap_missing_rate(title="MCAR"),
        md_mar.heatmap_missing_rate(title="MAR"),
        md_mnar.heatmap_missing_rate(title="MNAR")
        ],
    )

panel.show()

In [ ]:
md_mcar.barchart_intersection(title="MCAR").show()
md_mar.barchart_intersection(title="MAR").show()
md_mnar.barchart_intersection(title="MNAR").show()

this plot can help identify co-occurrence patterns between missing columns. with multiple columns having missing values across all three mechanisms, it provides a useful complement to the heatmap-based analysis.

the venn diagram below shows the same information in a set-overlap format, making it easier to identify which column combinations tend to be missing together.

In [ ]:
panel = Panel(
    plots=[
        md_mcar.barchart_venn(title="MCAR"),
        md_mar.barchart_venn(title="MAR"),
        md_mnar.barchart_venn(title="MNAR")
        ],
    title="Missing Value Overlap (Venn)",
    )

panel.show()

## Mechanism Discovery

we now attempt to prove each mechanism using visualizations. the strategy per case:

- MCAR: show that no observed variable predicts missingness in any column. missing values should be spread randomly across all rows and columns.
- MAR: show that observed values in some columns predict missingness in others. we expect `age` to explain missingness in `pclass` and `fare`, and `sibsp` to explain missingness in `parch`.
- MNAR: first, discard MAR by showing no observed variable consistently predicts missingness. then confirm that the pattern is driven by the columns' own unobserved values.

the following plot measures how strongly the observed values of each column can explain missingness in any other column.

In [ ]:
panel = Panel(
    plots=[
        md_mcar.heatmap_value_missingness_correlation(title="MCAR"),
        md_mar.heatmap_value_missingness_correlation(title="MAR"),
        md_mnar.heatmap_value_missingness_correlation(title="MNAR")
        ]
    )

panel.show()

- MCAR: All correlations are near zero. No observed variable can explain missingness in any other column. First evidence of MCAR.

- MAR: Three stronger values stand out, pointing to two candidate pairs: `age` could explain missingness in `pclass` and `fare`, and `sibsp` could explain missingness in `parch`. First evidence of MAR.

- MNAR: No strong values are visible, but the highest ones involve `sibsp` related to `age`, and `pclass` to `fare`. We will explore those relations further to rule out MAR before confirming MNAR.

we will now use the missingness heatmap to look for structural patterns that might confirm or challenge our initial assumptions.

In [ ]:
panel = Panel(
    plots=[
        md_mcar.heatmap_missingness(title="MCAR"),
        md_mar.heatmap_missingness(title="MAR"),
        md_mnar.heatmap_missingness(title="MNAR")
        ],
    title="Missingness Analysis by Mechanism",
    )

panel.show()

- MCAR: Missing values are spread across all columns with no visible pattern. Consistent with MCAR.

- MAR: Missing values in `parch` appear concentrated in a specific row segment, suggesting a structural pattern tied to another variable. The pattern is not fully conclusive here — we explore it in more detail below.

- MNAR: No clear pattern across variables. The distribution looks similar to MCAR visually, which rules out MAR. MNAR remains the candidate and we will explore it separately.

we focus the following plots on the MAR scenario, ordering the heatmap by `age` and then by `sibsp` to verify the two candidate relationships identified earlier.

In [ ]:
panel = Panel(
    plots=[
        md_mar.heatmap_missingness(
            title="MAR",
            order_by=[{"column": "age", "direction": "asc"}]
        ),
        md_mar.heatmap_missingness(
            title="MAR",
            order_by=[{"column": "sibsp", "direction": "asc"}]
        ),
    ],
    title="MAR Missingness Analysis",
    )

panel.show()

the first plot confirms a MAR relationship: when ordered by `age`, missing values in `pclass` and `fare` form a contiguous block, confirming that `age` explains their missingness and ruling out a random pattern.

the second plot confirms the other MAR relationship: missing values in `parch` are concentrated in rows with `sibsp=0`, confirming that `sibsp` explains missingness in `parch`. MAR confirmed for both pairs.

we now explore the MNAR scenario. the value-missingness heatmap suggested that `sibsp` might be related to `age` missingness, and `pclass` to `fare` missingness. we test whether those are genuine MAR signals or artefacts of the MNAR generation.

In [ ]:
panel = Panel(
    plots=[
        md_mnar.heatmap_missingness(
            title="MNAR",
            order_by=[{"column": "sibsp", "direction": "asc"}]
        ),
        md_mnar.heatmap_missingness(
            title="MNAR",
            order_by=[{"column": "pclass", "direction": "asc"}]
        ),
    ],
    title="MNAR Missingness Analysis",
    )

panel.show()

the first plot shows some concentration of `age` missing values at higher `sibsp` values, but the pattern is too spread to conclude MAR. missing values appear across the full range of `sibsp`.

the second plot shows slightly more `fare` missingness at higher `pclass` values, but again not a clear enough pattern to confirm MAR. neither plot supports a MAR explanation, which is consistent with MNAR — the missingness is driven by the columns' own unobserved values, not by any observed predictor.

In [ ]:
all_pclass = pd.concat([df_mcar["pclass"], df_mar["pclass"], df_mnar["pclass"]]).dropna()
all_age    = pd.concat([df_mcar["age"],    df_mar["age"],    df_mnar["age"]]).dropna()

age_min    = all_age.min()
age_offset = age_min - 0.12 * (all_age.max() - age_min)
yaxis_range = [age_offset - 3, all_age.max() * 1.05]
xaxis_range = [all_pclass.min() - 0.5, all_pclass.max() + 0.5]

panel = Panel(
    plots=[
        md_mcar.scatterplot_missingness(x="sibsp", y="age", title="MCAR",
            xaxis_range=xaxis_range, yaxis_range=yaxis_range),
        md_mar.scatterplot_missingness(x="age", y="fare", title="MAR",
            xaxis_range=xaxis_range, yaxis_range=yaxis_range),
        md_mnar.scatterplot_missingness(x="sibsp", y="age", title="MNAR",
            xaxis_range=xaxis_range, yaxis_range=yaxis_range),
        ],
    title="Missingness Analysis by Mechanism",
    )

panel.show()

- MCAR: Missing values are spread across all `sibsp` and `age` values with no pattern. MCAR confirmed.

- MAR: Missing `fare` values are concentrated at lower `age` values, visually confirming the MAR relationship.

- MNAR: Missing `age` values span the full range of `sibsp`. No predictor is visible. The distribution of present `age` values shows that lower ages are absent, which is the MNAR signature — the column's own unobserved values drive the missingness. MNAR confirmed.

## Comparative Discovery

to confirm MNAR, we compare the observed distribution of `age` and `fare` between MCAR and MNAR. in MCAR the full range of values survives; in MNAR the lower tail is absent because those values were removed. the truncation is the direct evidence of MNAR.

In [ ]:
panel = Panel(
    plots=[
        md_mcar.density_missingness(x="age", color_by="fare", title="MCAR"),
        md_mnar.density_missingness(x="age", color_by="fare", title="MNAR"),
    ],
    title="Observed Age Distribution: MCAR vs MNAR",
)

panel.show()

In [ ]:
panel = Panel(
    plots=[
        md_mcar.density_missingness(x="fare", color_by="age", title="MCAR"),
        md_mnar.density_missingness(x="fare", color_by="age", title="MNAR"),
    ],
    title="Observed Fare Distribution: MCAR vs MNAR",
)

panel.show()

- MCAR: both curves span the full range of values from near zero upwards, with the two groups (present and missing) overlapping closely. no truncation visible.

- MNAR (`age`): the distribution is clearly truncated — both curves start at ~20, with no values below that threshold. in MCAR the same distribution starts near zero. this is direct evidence that the missing age values came from the lower end of the distribution, the defining characteristic of MNAR.

- MNAR (`fare`): the truncation is not clearly visible in the density plot. fare is already heavily right-skewed, so removing the lowest values does not produce a noticeable shift in the curve shape. the age plot remains the stronger evidence for MNAR in this dataset.

the heatmaps confirm the MAR pairs and rule them out for MNAR. we now use boxplot and density plots to further validate each mechanism.

In [ ]:
# we'll use different columns for this plot to showcase the difference in the distribution on each case previously showed

panel = Panel(
    plots=[
        md_mcar.boxplot_missingness(x="sibsp", color_by="age", title="MCAR"),
        md_mar.boxplot_missingness(x="age", color_by="fare", title="MAR"),
        md_mnar.boxplot_missingness(x="sibsp", color_by="age", title="MNAR")
        ],
    title="Missingness Analysis by Mechanism",
    )

panel.show()

- MCAR: The distribution of `sibsp` is similar between present and missing `age` groups, demonstrating no pattern. Consistent with MCAR.

- MAR: Missing `fare` values are concentrated at lower `age` values, confirming the MAR relationship identified earlier.

- MNAR: Missing `age` values are spread across all `sibsp` values with no clear pattern. This rules out both MCAR and MAR. MNAR is the remaining explanation.

In [ ]:
panel = Panel(
    plots=[
        md_mcar.density_missingness(x="sibsp", color_by="age", title="MCAR"),
        md_mar.density_missingness(x="age", color_by="fare", title="MAR"),
        md_mnar.density_missingness(x="sibsp", color_by="age", title="MNAR")
        ],
    title="Missingness Analysis by Mechanism",
    )

panel.show()

- MCAR: The density curves for present and missing `age` overlap closely across all `sibsp` values, with only minor divergence. No pattern. MCAR confirmed.

- MAR: The missing `fare` density is sharply concentrated at lower `age` values, while the present curve spans a much broader range. Clear MAR signal.

- MNAR: The missing `age` density is more concentrated at `sibsp=0` compared to the present curve, which spreads into higher `sibsp` values. This divergence is a side effect of removing lower age values — younger passengers tend to have fewer siblings. It does not indicate a MAR predictor. Discarding MCAR (the removal is not random) and MAR (no observed variable causally predicts it), MNAR is confirmed.

## Conclusions

- MCAR: No observed variable explains missingness in any column. Missing values are spread randomly across all columns and all rows, with no structural pattern visible in any of the plots. MCAR confirmed.

- MAR: Two MAR relationships are confirmed. `age` explains missingness in `fare` and `pclass` — lower age values predict those columns being missing. `sibsp` explains missingness in `parch` — missingness concentrates at the lowest `sibsp` values (sibsp=0). Both relationships were detected by the value-missingness heatmap and confirmed by the ordered missingness heatmaps and the distributional plots.

- MNAR: No observed variable consistently predicts missingness in `age` or `fare`. The mild correlations with `sibsp` and `pclass` seen in the heatmap are not strong enough to conclude MAR, and the distributional plots show no clear predictor. Since the generator removed lower-end values of `age` and `fare`, the missingness is driven by the columns' own unobserved values — a defining characteristic of MNAR.